# Simulación del estómago — Fase 7: representación 2D de la digestión

Una película de **4 horas de digestión** de la comida de consenso (255 kcal, 315 mL), en la que se ve:
1. **La ingesta**: el estómago empieza en ayunas, con 25 mL de jugo, y la comida baja por el esófago en 18 tragos durante 6 min. El fundus y el cuerpo se relajan para alojarla.
2. **Los movimientos peristálticos**: las ondas nacen a mitad del estómago, se hacen más profundas en el antro y viajan hacia el píloro, donde trituran las partículas.
3. **La dilución progresiva**: el jugo entra por las paredes, diluye la comida desde fuera hacia dentro, la acidifica y sale con ella por el píloro hacia el duodeno.

**Qué es simulación y qué es dibujo.** Todos los valores que se ven salen del **modelo integrado de la fase 6**: la fracción de comida en el líquido, el pH, las partículas, el volumen y lo que pasa el píloro. Lo único nuevo en la física es la entrada de la comida y un trazador que la sigue (sección 1). La **forma** del estómago es un dibujo: la losa rectangular del modelo se proyecta sobre una "J" anatómica, y la deformación de la pared por las ondas también se dibuja (sección 2).

**Tiempo en Colab**: unos 3 minutos en CPU. Casi todo se va en guardar el vídeo.

## 1. Lo que se añade al modelo integrado

**Estado inicial en ayunas.** El estómago tiene $V_0 = 25$ mL de jugo a pH 1,5, sin comida y sin partículas.

**Entrada de la comida.** Entra un caudal constante durante $t_\text{ing} = 6$ min por el tramo del cardias, que es la fila de celdas de la curvatura menor con $0{,}4 < x < 1{,}6$ cm:
$$q_c(t) = \frac{V_\text{comida}}{t_\text{ing}}\;\mathbb 1[t < t_\text{ing}], \qquad \frac{dV}{dt} = q_\text{jugo}(t) + q_c(t) - Q_\text{píloro}(t).$$
El líquido de la comida trae su tampón ($B_c$) y sus protones ($T_c$) al pH inicial de la comida. Las partículas sólidas entran en tragos discretos: cada una tiene asignado uno de los 18 tragos.

**Trazador de comida $M$** (la magnitud que representa la dilución). Es la fracción del líquido que procede de la comida: $M = 1$ es comida pura y $M = 0$ es jugo puro. Se transporta como cualquier otra especie del modelo:
$$\frac{\partial (M\,h)}{\partial t} + \nabla\!\cdot\!(\mathbf u_b\, M\,h) = \nabla\!\cdot\!\big(h\,\mathbf D(x)\,\nabla M\big) + \frac{q_c}{A_\text{card}}\,\mathbb 1_\text{cardias}.$$
- $h$ es el espesor local de la losa.
- $\mathbf u_b$ es el flujo neto que sale del balance de volumen, con el flujo transversal del jugo que entra por las paredes.
- $\mathbf D(x)$ es la dispersión medida con LBM en la fase 6.

El jugo que entra no lleva comida, así que **diluye** $M$, y el píloro se lleva $M$ con el líquido que sale. Comprobaciones: $\int M\,dV + M_\text{salido} = V_\text{comida ingerida}$ (V1) y la masa de sólido se conserva (V2).

**Un cambio numérico respecto a la fase 6.** Durante la ingesta el estómago crece y el flujo neto puede ir **hacia el fundus** en algunas caras. Por eso el esquema *upwind* elige ahora la celda aguas arriba según el signo del caudal en cada cara. En la fase 6 el caudal siempre iba hacia el píloro y los dos esquemas coinciden.

## 2. Cómo se dibuja: de la losa del modelo a la anatomía

La losa del modelo tiene $x \in [0, 10]$ cm, del fondo al píloro, e $y \in [0, 4]$ cm. Se proyecta sobre una línea media curva $\mathbf c(s)$, un *spline* por puntos que dibujan una "J":
$$\mathbf P(x, y) = \mathbf c(x) + \Big(\frac{y}{L_y} - \tfrac12\Big)\, w(x, t)\; \mathbf n(x).$$
- $\mathbf n$ es la normal hacia la **curvatura menor**, donde está el cardias. Así $y = 0$ queda en la **curvatura mayor**, que es donde sedimentan las partículas.
- La **anchura** $w$ combina tres efectos:
$$w(x,t) = w_0(x)\; f_V\big(x, V(t)\big)\;\big(1 - \phi(x,t)\big).$$
  - $w_0(x)$ es la forma anatómica: fundus ancho, antro estrecho.
  - $f_V$ es la **relajación receptiva**: el fundus y el cuerpo se distienden con el volumen ($\propto (V/V_\text{max})^{0.6}$) y el antro casi no cambia de calibre.
  - $\phi$ son las **ondas peristálticas**, con los mismos parámetros que el flujo LBM de la fase 3:
$$\phi(x,t) = \phi_\text{max}\; \text{rampa}(x)\sum_k \exp\!\Big[-\Big(\frac{x - x_\text{ini} - c\,(t \bmod T) - k\,cT}{\sigma}\Big)^2\Big].$$
  Aquí $c = 0{,}25$ cm/s, $T = 20$ s y la rampa va de 0 en $x = 4{,}5$ a 1 en $x = 6{,}5$ cm.
- **Fundus**: el extremo $x < 0$ se cierra con una cúpula que toma los valores de la primera columna del modelo.
- **Partículas**: se dibujan apoyadas en la curvatura mayor, formando un montón más alto donde hay más.
- **Duodeno**: muestra el color del líquido que sale por el píloro y las partículas que lo cruzaron en los últimos 8 min.

**Cámara rápida.** Una onda dura 20 s y la digestión, 4 horas. La película va casi en tiempo real durante la ingesta (×50, 1 fotograma cada 5 s), con las ondas en su fase real. Después acelera (×300 y ×1200, 1 fotograma cada 2 min). Como en 2 min caben 6 ondas, mostrar su fase real congelaría la imagen, así que a partir de ahí **la fase de las ondas es ilustrativa**: avanza 5 s por fotograma, igual que en la ingesta. El modelo no usa esta fase. Usa el efecto promediado de las ondas: dispersión y molienda medidas con LBM.

## 3. Código

In [ ]:
import math, time, os, base64, pickle
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
    "animation.embed_limit": 80,
})
GRIS = "#3b3b3b"; GRIS_MEDIO = "#9a9a96"; TINTA2 = "#52514e"
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"
os.makedirs("figuras", exist_ok=True)
VERIFICACION = []

### 3.1 Flujo LBM y mapas de la escala rápida (fases 3 y 6)

In [ ]:
CODIGO_LBM = r'''
# ===========================================================================
# Lattice Boltzmann D2Q9 escrito con funciones PURAS (sin modificar arrays en el sitio).
# Así el mismo código corre con NumPy, CuPy (GPU, misma API) o jax.numpy (GPU + jit).
# Unidades de red: dx = dt = 1, c_s^2 = 1/3, nu = (tau - 1/2)/3.
# Convenio de arrays: f tiene forma (9, ny, nx); la fila j = 0 es el fondo (y crece hacia arriba).
# ===========================================================================
CX  = [0, 1, 0, -1, 0, 1, -1, -1, 1]
CY  = [0, 0, 1, 0, -1, 1, 1, -1, -1]
W   = [4 / 9] + [1 / 9] * 4 + [1 / 36] * 4
OPP = [0, 3, 4, 1, 2, 7, 8, 5, 6]          # dirección opuesta


class Red:
    # Constantes D2Q9 ya convertidas al backend (xp) y al tipo de dato elegidos
    def __init__(self, xp, dtype="float64"):
        self.xp, self.dtype = xp, dtype
        self.cx = xp.asarray(CX, dtype=dtype).reshape(9, 1, 1)
        self.cy = xp.asarray(CY, dtype=dtype).reshape(9, 1, 1)
        self.w = xp.asarray(W, dtype=dtype).reshape(9, 1, 1)


def equilibrio(rho, ux, uy, R):
    cu = 3.0 * (R.cx * ux + R.cy * uy)
    usq = 1.5 * (ux * ux + uy * uy)
    return R.w * rho * (1.0 + cu + 0.5 * cu * cu - usq)


def macroscopicas(f, R, Fx=0.0, Fy=0.0):
    rho = f.sum(axis=0)
    jx = f[1] - f[3] + f[5] - f[6] - f[7] + f[8]        # sum_i f_i c_ix, escrito explícitamente (más rápido)
    jy = f[2] - f[4] + f[5] + f[6] - f[7] - f[8]
    ux = (jx + 0.5 * Fx) / rho                          # desplazamiento de Guo: u = (sum f c + F/2)/rho
    uy = (jy + 0.5 * Fy) / rho
    return rho, ux, uy


def colision_bgk(f, rho, ux, uy, tau, R, Fx=0.0, Fy=0.0):
    feq = equilibrio(rho, ux, uy, R)
    cu = R.cx * ux + R.cy * uy
    cF = R.cx * Fx + R.cy * Fy
    # término de fuerza de Guo (2002): segundo orden con fuerza volumétrica
    S = (1.0 - 0.5 / tau) * R.w * (3.0 * ((R.cx - ux) * Fx + (R.cy - uy) * Fy) + 9.0 * cu * cF)
    return f - (f - feq) / tau + S


def propagar(f, R):
    xp = R.xp
    return xp.stack([xp.roll(f[i], shift=(CY[i], CX[i]), axis=(0, 1)) for i in range(9)])


def rebote(fs, fpost, solido, uwx, uwy, rho, R):
    # Rebote a mitad de enlace con pared móvil (Ladd):
    # si la población i llega a un nodo fluido x desde un nodo sólido x - c_i, se sustituye por
    # f_i(x) = f*_opp(i)(x) + 6 w_i rho (c_i · u_w),  con u_w la velocidad de la pared en ese nodo sólido.
    xp = R.xp
    salida = []
    fluido = ~solido
    for i in range(9):
        if i == 0:
            salida.append(fs[0]); continue
        desde_solido = xp.roll(solido, shift=(CY[i], CX[i]), axis=(0, 1)) & fluido
        cu_w = 0.0
        if CX[i] != 0 and uwx is not None:
            cu_w = cu_w + CX[i] * xp.roll(uwx, shift=(CY[i], CX[i]), axis=(0, 1))
        if CY[i] != 0 and uwy is not None:
            cu_w = cu_w + CY[i] * xp.roll(uwy, shift=(CY[i], CX[i]), axis=(0, 1))
        rebotado = fpost[OPP[i]] + 6.0 * W[i] * rho * cu_w
        salida.append(xp.where(desde_solido, rebotado, fs[i]))
    return xp.stack(salida)


def rellenar(f, solido_viejo, solido_nuevo, rho, uwx, uwy, R):
    # Nodos que pasan de sólido a fluido ("nodos frescos") se inicializan con el equilibrio:
    # densidad = media de los vecinos fluidos, velocidad = la de la pared (Lallemand y Luo, 2003).
    xp = R.xp
    fresco = solido_viejo & ~solido_nuevo
    fluido_previo = (~solido_viejo).astype(rho.dtype)
    suma, n = 0.0 * rho, 0.0 * rho
    for i in range(1, 9):
        suma = suma + xp.roll(rho * fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
        n = n + xp.roll(fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
    rho_vec = xp.where(n > 0, suma / xp.maximum(n, 1.0), 1.0)
    cero = 0.0 * rho
    f_fresco = equilibrio(rho_vec, cero if uwx is None else uwx, cero if uwy is None else uwy, R)
    f = xp.where(fresco[None], f_fresco, f)
    # dentro del sólido guardamos el equilibrio en reposo, f_i = w_i (no interviene en la dinámica)
    return xp.where(solido_nuevo[None], R.w + 0.0 * f, f)


def crear_paso(R, tau, geometria, Fx=0.0, Fy=0.0):
    # Devuelve paso(f, t) -> f(t+1). geometria(t) -> (solido, uwx, uwy) en unidades de red.
    def paso(f, t):
        solido, uwx, uwy = geometria(t)
        rho, ux, uy = macroscopicas(f, R, Fx, Fy)
        fpost = colision_bgk(f, rho, ux, uy, tau, R, Fx, Fy)
        fs = propagar(fpost, R)
        fs = rebote(fs, fpost, solido, uwx, uwy, rho, R)
        solido2, uwx2, uwy2 = geometria(t + 1)
        return rellenar(fs, solido, solido2, rho, uwx2, uwy2, R)
    return paso


def tension_viscosa(f, tau, R, Fx=0.0, Fy=0.0):
    # Tensor de tensiones viscosas a partir de la parte de no equilibrio (sin derivar el campo):
    # sigma_ab = -(1 - 1/(2 tau)) sum_i (f_i - feq_i) c_ia c_ib    [unidades de red]
    rho, ux, uy = macroscopicas(f, R, Fx, Fy)
    fneq = f - equilibrio(rho, ux, uy, R)
    k = -(1.0 - 0.5 / tau)
    sxx = k * (fneq * R.cx * R.cx).sum(axis=0)
    syy = k * (fneq * R.cy * R.cy).sum(axis=0)
    sxy = k * (fneq * R.cx * R.cy).sum(axis=0)
    return sxx, syy, sxy


def geometria_fija(solido, R):
    xp = R.xp
    cero = xp.zeros(solido.shape, dtype=R.dtype)
    return lambda t: (solido, cero, cero)


# ---------------------------------------------------------------------------
# Paredes que se mueven: canal peristáltico periódico (test de Shapiro)
# ---------------------------------------------------------------------------
def geometria_peristaltica(nx, ny, a, phi, lam, c, R):
    # Semiancho H(x,t) = a (1 + phi sin(2 pi (x - c t)/lam)) alrededor del eje y = ny/2.
    # Las paredes se mueven solo en vertical (como en Shapiro et al., 1969).
    xp = R.xp
    x = xp.arange(nx, dtype=R.dtype)[None, :] + 0.5
    y = xp.arange(ny, dtype=R.dtype)[:, None] + 0.5
    yc = ny / 2.0
    k = 2 * math.pi / lam
    def geom(t):
        H = a * (1.0 + phi * xp.sin(k * (x - c * t)))
        dHdt = -a * phi * k * c * xp.cos(k * (x - c * t))
        abajo, arriba = y < yc - H, y > yc + H
        solido = abajo | arriba
        uwy = xp.where(abajo, -dHdt, 0.0 * y) + xp.where(arriba, dHdt, 0.0 * y)
        return solido, None, uwy                        # uwx = None: las paredes solo se mueven en vertical
    return geom


def integrar(paso, f, t0, n_pasos, R, usar_jax=False):
    # Bucle de tiempo. Con JAX se compila con lax.fori_loop (todo el bucle en la GPU).
    if usar_jax:
        from jax import lax
        f, _ = lax.fori_loop(0, n_pasos, lambda i, ft: (paso(ft[0], ft[1]), ft[1] + 1.0), (f, R.xp.asarray(t0, dtype=R.dtype)))
        return f
    for n in range(n_pasos):
        f = paso(f, t0 + n)
    return f

@dataclass
class ParamAntro:
    # Geometría y movimiento (unidades físicas: cm, s)
    Lx: float = 10.0          # cm   longitud (fondo -> píloro)
    H: float = 4.0            # cm   anchura en reposo
    margen: float = 1.0       # cm   holgura para que el fondo pueda dilatarse
    x_ini: float = 4.5        # cm   donde empiezan las ondas antrales
    x_plena: float = 6.5      # cm   donde alcanzan su amplitud máxima
    L_fondo: float = 3.5      # cm   zona proximal que se dilata para conservar el volumen
    A_max: float = 1.4        # cm   profundidad de la contracción en cada pared (70 % de oclusión)
    w: float = 0.5            # cm   semiancho de la onda
    c: float = 0.25           # cm/s velocidad de la onda peristáltica
    T: float = 20.0           # s    periodo (3 ondas por minuto)
    nu: float = 0.1           # cm²/s viscosidad cinemática (comida líquida espesa; agua = 0.01)
    rho: float = 1.0          # g/cm³
    # Discretización
    nx: int = 128             # nodos a lo largo (sin contar las paredes de los extremos)
    tau: float = 0.65


class Unidades:
    # Conversión física <-> red: dx fija la malla; dt sale de igualar la viscosidad.
    def __init__(self, pa):
        self.dx = pa.Lx / pa.nx                        # cm
        self.nu_red = (pa.tau - 0.5) / 3.0
        self.dt = self.nu_red * self.dx ** 2 / pa.nu    # s
        self.vel = self.dx / self.dt                    # (cm/s) por unidad de red
        self.tension = pa.rho * self.vel ** 2           # (dyn/cm² = 0.1 Pa) por unidad de red

    def resumen(self, pa):
        return {"dx [cm]": self.dx, "dt [s]": self.dt, "pasos por onda": pa.T / self.dt,
                "c onda [red]": pa.c / self.vel, "Re = c (H/2)/nu": pa.c * (pa.H / 2) / pa.nu,
                "nu [red]": self.nu_red}


def geometria_antro(pa, R):
    # Paredes superior e inferior con ondas que viajan hacia el píloro (x = Lx).
    # El fondo (x < L_fondo) se dilata lo justo para que el volumen total no cambie.
    # Extremos x = 0 (fondo) y x = Lx (píloro, cerrado en esta fase) son paredes fijas.
    xp = R.xp
    U = Unidades(pa)
    dx = U.dx
    ny = int(round((pa.H + 2 * pa.margen) / dx))
    nx = pa.nx + 2                                       # + una columna de pared en cada extremo
    x = (xp.arange(nx, dtype=R.dtype)[None, :] - 0.5) * dx          # cm (la columna 0 es pared)
    y = (xp.arange(ny, dtype=R.dtype)[:, None] + 0.5) * dx
    rampa = xp.clip((x - pa.x_ini) / (pa.x_plena - pa.x_ini), 0.0, 1.0)
    envolvente = pa.A_max * rampa * rampa * (3 - 2 * rampa)         # suave (smoothstep)
    ventana_fondo = xp.clip((pa.L_fondo + 1.0 - x) / 1.0, 0.0, 1.0) * (x > 0) * (x < pa.Lx)
    int_ventana = float(xp.sum(ventana_fondo)) * dx
    x0 = pa.x_ini - 3 * pa.w
    K = int(math.ceil((pa.Lx + 3 * pa.w - x0) / (pa.c * pa.T))) + 1
    extremos = (x <= 0) | (x >= pa.Lx)
    def ondas(t_fis):
        s = t_fis % pa.T
        d = 0.0 * x
        dd = 0.0 * x
        for k in range(K):
            xk = x0 + pa.c * (s + k * pa.T)
            g = xp.exp(-((x - xk) / pa.w) ** 2)
            d = d + g
            dd = dd + g * 2 * (x - xk) * pa.c / pa.w ** 2
        return envolvente * d, envolvente * dd                      # delta [cm], d delta/dt [cm/s]
    def geom(t):
        t_fis = t * U.dt
        delta, ddelta = ondas(t_fis)
        interior = (x > 0) & (x < pa.Lx)
        area = xp.sum(delta * interior) * dx                         # área contraída por pared
        darea = xp.sum(ddelta * interior) * dx
        d_f = area / int_ventana                                     # dilatación del fondo
        dd_f = darea / int_ventana
        yb = pa.margen + delta - ventana_fondo * d_f
        yt = pa.margen + pa.H - delta + ventana_fondo * d_f
        vb = ddelta - ventana_fondo * dd_f                           # cm/s
        vt = -ddelta + ventana_fondo * dd_f
        abajo, arriba = y < yb, y > yt
        solido = abajo | arriba | extremos
        uwy = (xp.where(abajo, vb, 0.0 * y) + xp.where(arriba, vt, 0.0 * y)) * ~extremos / U.vel
        return solido, None, uwy
    return geom, nx, ny, U

def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11


def simular_antro(pa, R, periodos=4, guardar_cada_s=0.25, n_particulas=3000, semilla=1,
                  usar_jax=False, informar=True):
    # Ejecuta el antro y devuelve instantáneas (en NumPy y unidades físicas) + trayectorias de trazadores.
    xp = R.xp
    geom, nx, ny, U = geometria_antro(pa, R)
    paso = crear_paso(R, pa.tau, geom)
    f = R.w * xp.ones((9, ny, nx), dtype=R.dtype)                     # reposo, rho = 1
    pasos_bloque = max(1, int(round(guardar_cada_s / U.dt)))
    n_bloques = int(round(periodos * pa.T / (pasos_bloque * U.dt)))
    if usar_jax:
        import jax
        avanzar = jax.jit(lambda f, t0: integrar(paso, f, t0, pasos_bloque, R, usar_jax=True))
    else:
        avanzar = lambda f, t0: integrar(paso, f, t0, pasos_bloque, R)
    # trazadores: repartidos al azar por el fluido inicial (coordenadas de red)
    rng = np.random.default_rng(semilla)
    s0 = a_numpy(geom(0)[0])
    cand = np.argwhere(~s0)
    elegido = cand[rng.choice(len(cand), n_particulas, replace=False)]
    py = elegido[:, 0] + rng.uniform(-0.5, 0.5, n_particulas)
    px = elegido[:, 1] + rng.uniform(-0.5, 0.5, n_particulas)
    hist = {k: [] for k in ("t", "ux", "uy", "rho", "solido", "tau_max", "px", "py", "masa")}
    def campos(f, t):
        rho, ux, uy = macroscopicas(f, R)
        sxx, syy, sxy = tension_viscosa(f, pa.tau, R)
        tmax = xp.sqrt(0.25 * (sxx - syy) ** 2 + sxy ** 2)            # cizalla máxima
        sol = geom(t)[0]
        return [a_numpy(v) for v in (rho, ux, uy, tmax, sol)]
    t_red = 0
    rho, ux, uy, tmax, sol = campos(f, t_red)
    t_ini = time.time()
    for b in range(n_bloques + 1):
        hist["t"].append(t_red * U.dt)
        hist["ux"].append(np.where(sol, np.nan, ux) * U.vel)
        hist["uy"].append(np.where(sol, np.nan, uy) * U.vel)
        hist["rho"].append(np.where(sol, np.nan, rho))
        hist["tau_max"].append(np.where(sol, np.nan, tmax) * U.tension / 10.0)   # Pa
        hist["solido"].append(sol)
        hist["px"].append(px.copy()); hist["py"].append(py.copy())
        hist["masa"].append(float(np.sum(np.where(sol, 0.0, rho))))
        if b == n_bloques:
            break
        f = avanzar(f, float(t_red))
        t_red += pasos_bloque
        rho2, ux2, uy2, tmax2, sol2 = campos(f, t_red)
        # trazadores: Heun con el campo al principio y al final del bloque (desplazamiento en nodos)
        n_p = pasos_bloque
        vx1 = interpolar_bilineal(np.where(sol, 0, ux), px, py); vy1 = interpolar_bilineal(np.where(sol, 0, uy), px, py)
        qx, qy = px + n_p * vx1, py + n_p * vy1
        vx2 = interpolar_bilineal(np.where(sol2, 0, ux2), qx, qy); vy2 = interpolar_bilineal(np.where(sol2, 0, uy2), qx, qy)
        nx_, ny_ = px + 0.5 * n_p * (vx1 + vx2), py + 0.5 * n_p * (vy1 + vy2)
        ii = np.clip(np.rint(nx_).astype(int), 0, nx - 1); jj = np.clip(np.rint(ny_).astype(int), 0, ny - 1)
        dentro = ~sol2[jj, ii]
        px = np.where(dentro, nx_, px); py = np.where(dentro, ny_, py)   # si caería en la pared, se queda
        rho, ux, uy, tmax, sol = rho2, ux2, uy2, tmax2, sol2
        if informar and (b + 1) % max(1, n_bloques // 8) == 0:
            print(f"  t = {t_red * U.dt:6.1f} s   ({time.time() - t_ini:5.1f} s de cálculo)")
    out = {k: np.array(v) for k, v in hist.items()}
    out["U"], out["nx"], out["ny"], out["pasos"] = U, nx, ny, t_red
    out["segundos"] = time.time() - t_ini
    return out



'''

In [ ]:
def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11

In [ ]:
class CampoPeriodico:
    # Campo de velocidades de un periodo de onda (fase 3), repetido periódicamente en el tiempo.
    def __init__(self, datos):
        self.t = datos["t"]; self.ux = datos["ux"]; self.uy = datos["uy"]; self.sol = datos["solido"]
        self.x = datos["x"]; self.y = datos["y"]; self.dx = float(datos["dx"]); self.T = float(datos["periodo"])
        self.nu = float(datos["nu"])
        self.tau = datos["tau_max"] if "tau_max" in datos else None
        self.nf = len(self.t) - 1                    # el último marco repite el primero (t = T)
        self.dtf = self.t[1] - self.t[0]
        # hueco local (cm) en cada columna y marco: nº de nodos fluidos × dx
        self.hueco = (~self.sol).sum(axis=1) * self.dx

    def _marco(self, t):
        s = (t % self.T) / self.dtf
        k = int(np.floor(s)) % self.nf
        return k, s - np.floor(s)

    def velocidad(self, t, px, py):
        k, a = self._marco(t)
        u0 = self._interp(self.ux[k], px, py); u1 = self._interp(self.ux[k + 1], px, py)
        v0 = self._interp(self.uy[k], px, py); v1 = self._interp(self.uy[k + 1], px, py)
        return (1 - a) * u0 + a * u1, (1 - a) * v0 + a * v1

    def _indices(self, px, py):
        fx = (px - self.x[0]) / self.dx; fy = (py - self.y[0]) / self.dx
        return fx, fy

    def _interp(self, campo, px, py):
        fx, fy = self._indices(px, py)
        return interpolar_bilineal(campo, fx, fy)

    def solido_en(self, t, px, py):
        k, a = self._marco(t)
        sol = self.sol[k if a < 0.5 else k + 1]
        fx, fy = self._indices(px, py)
        i = np.clip(np.rint(fx).astype(int), 0, sol.shape[1] - 1); j = np.clip(np.rint(fy).astype(int), 0, sol.shape[0] - 1)
        return sol[j, i], sol, i, j

    def hueco_en(self, t, px):
        k, _ = self._marco(t)
        fx = (px - self.x[0]) / self.dx
        i = np.clip(np.rint(fx).astype(int), 0, self.hueco.shape[1] - 1)
        return self.hueco[k, i]


def tiempo_relajacion(d_cm, rho_p, rho_f, nu, v_rel):
    # Arrastre de Stokes con corrección de Schiller–Naumann: tau = rho_p d² / (18 mu f(Re_p))
    mu = rho_f * nu
    Re_p = np.abs(v_rel) * d_cm / nu
    f = 1 + 0.15 * Re_p ** 0.687
    return rho_p * d_cm ** 2 / (18 * mu * f)


def seguir_particulas(campo, d_mm, rho_p, rho_f=1.0, g=981.0, t_final=200.0, dt=0.02, semilla=0,
                      x_zona=8.0, frac_cuello=0.6, guardar_cada=0.5):
    # Partículas con inercia y sedimentación (gravedad hacia -y) en el campo periódico.
    # Integración exponencial con el fluido congelado durante dt (exacta para arrastre lineal).
    # Evento de compresión: la partícula está en el antro terminal (x > x_zona) dentro de un cuello
    # con hueco < frac_cuello · H. Se cuenta una vez por paso de onda.
    rng = np.random.default_rng(semilla)
    d_mm = np.asarray(d_mm, float); n = d_mm.size; d_cm = d_mm / 10
    s0 = campo.sol[0]
    cand = np.argwhere(~s0)
    sel = cand[rng.choice(len(cand), n, replace=True)]
    py = campo.y[sel[:, 0]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    px = campo.x[sel[:, 1]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    vx, vy = campo.velocidad(0.0, px, py)
    flot = (1 - rho_f / rho_p) * g
    cuerpo = (campo.x > 3.8) & (campo.x < 4.4)                              # cuerpo gástrico: sin ondas ni dilatación
    H_ref = float(np.median(campo.hueco[:, cuerpo]))
    ultimo_evento = np.full(n, -1e9)
    eventos_x = []
    eventos = np.zeros(n)
    tiempo_zona = np.zeros(n)
    hist_x, hist_y, hist_t = [px.copy()], [py.copy()], [0.0]
    cada = max(1, int(round(guardar_cada / dt)))
    for paso in range(int(round(t_final / dt))):
        t = paso * dt
        ux, uy = campo.velocidad(t, px, py)
        rx, ry = vx - ux, vy - uy
        tau = tiempo_relajacion(d_cm, rho_p, rho_f, campo.nu, np.hypot(rx, ry))
        vtx, vty = 0.0, -tau * flot                                          # velocidad terminal relativa
        e = np.exp(-dt / tau)
        cx_, cy_ = vx - ux - vtx, vy - uy - vty
        nx_ = px + (ux + vtx) * dt + cx_ * tau * (1 - e)
        ny_ = py + (uy + vty) * dt + cy_ * tau * (1 - e)
        vx = ux + vtx + cx_ * e; vy = uy + vty + cy_ * e
        # paredes: si el centro entra en el sólido, se recoloca en el nodo fluido más cercano de su columna
        dentro, sol, i, j = campo.solido_en(t + dt, nx_, ny_)
        if dentro.any():
            for q in np.where(dentro)[0]:
                col = ~sol[:, i[q]]
                if col.any():
                    jj = np.where(col)[0]
                    jn = jj[np.argmin(np.abs(jj - j[q]))]
                    ny_[q] = campo.y[jn]
                else:
                    nx_[q], ny_[q] = px[q], py[q]
                vy[q] = 0.0
        px, py = nx_, ny_
        # eventos de compresión en el antro terminal
        en_zona = px > x_zona
        tiempo_zona += en_zona * dt
        cuello = en_zona & (campo.hueco_en(t + dt, px) < frac_cuello * H_ref)
        nuevo = cuello & (t - ultimo_evento > 0.5 * campo.T)
        eventos += nuevo; ultimo_evento = np.where(nuevo, t, ultimo_evento)
        eventos_x.extend(px[nuevo].tolist())
        if (paso + 1) % cada == 0:
            hist_x.append(px.copy()); hist_y.append(py.copy()); hist_t.append(t + dt)
    return {"x": np.array(hist_x), "y": np.array(hist_y), "t": np.array(hist_t), "d_mm": d_mm,
            "eventos_por_min": eventos / (t_final / 60.0), "frac_tiempo_zona": tiempo_zona / t_final,
            "eventos_x": np.array(eventos_x)}

In [ ]:
def mapas_escala_rapida(campo, ondas=6, n=1500, d_part=3.0, rho_part=1.03, ancho_bin=1.0, margen=1.0, semilla=0, ventana=2):
    # Escala rápida -> mapas promediados en fase para el bucle lento.
    # 1) Trazadores de fluido: dispersión axial y transversal D_x(x), D_y(x) y deriva media u(x).
    # 2) Partículas densas (sedimentan): dispersión axial en la capa del fondo y frecuencia de molienda λ_g(x).
    T = campo.T
    dt_g = 0.5                                                   # s entre posiciones guardadas
    paso_onda = int(round(T / dt_g))
    bordes = np.arange(0.0, 10.0 + 1e-9, ancho_bin); centros = 0.5 * (bordes[1:] + bordes[:-1]); nb = len(centros)
    res = {"x": centros, "bordes": bordes, "T": T}
    for tipo, d, rp in (("fluido", 0.01, 1.0), ("particula", d_part, rho_part)):
        L = seguir_particulas(campo, np.full(n, d), rp, t_final=ondas * T, dt=0.02, guardar_cada=dt_g, semilla=semilla)
        X, Y = L["x"], L["y"] - margen
        idx = np.arange(0, X.shape[0], paso_onda)
        dxs, dys, x0s = [], [], []
        for a, b in zip(idx[:-ventana], idx[ventana:]):            # incrementos de `ventana` ondas (deslizantes)
            dxs.append(X[b] - X[a]); dys.append(Y[b] - Y[a]); x0s.append(X[a])
        dxs, dys, x0s = np.concatenate(dxs), np.concatenate(dys), np.concatenate(x0s)
        cual = np.clip(np.digitize(x0s, bordes) - 1, 0, nb - 1)
        Tv = ventana * T / 60                                                   # min
        # D a partir del desplazamiento cuadrático medio (incluye la dispersión por cizalla de la deriva media)
        Dx = np.array([np.mean(dxs[cual == i] ** 2) / (2 * Tv) if np.sum(cual == i) > 10 else np.nan for i in range(nb)])
        Dy = np.array([np.mean(dys[cual == i] ** 2) / (2 * Tv) if np.sum(cual == i) > 10 else np.nan for i in range(nb)])
        ux = np.array([np.mean(dxs[cual == i]) / Tv if np.sum(cual == i) > 10 else np.nan for i in range(nb)])
        rellenar = lambda v: np.interp(centros, centros[np.isfinite(v)], v[np.isfinite(v)])
        res[tipo] = {"Dx": np.maximum(rellenar(Dx), 1e-3), "Dy": np.maximum(rellenar(Dy), 1e-3), "u_media": rellenar(ux),
                     "x_tray": X, "y_tray": Y, "t_tray": L["t"]}
        if tipo == "particula":
            # tiempo pasado en cada franja (min) y eventos de molienda en cada franja
            cual_t = np.clip(np.digitize(X[1:], bordes) - 1, 0, nb - 1)
            tiempo = np.array([np.sum(cual_t == i) for i in range(nb)]) * dt_g / 60
            ev = np.histogram(L["eventos_x"], bordes)[0]
            res[tipo]["lambda_g"] = np.where(tiempo > 0, ev / np.maximum(tiempo, 1e-9), 0.0)
            res[tipo]["altura_media"] = float(np.mean(Y[-1]))
    return res


def paseo_aleatorio(x0, Dx_tab, x_tab, T_min, n_ondas, sub=40, semilla=0, L=10.0):
    # Paseo aleatorio de Itô con D(x) variable: dx = D'(x) dt + sqrt(2 D(x) dt) ξ  (forma conservativa), reflexión en 0 y L
    rng = np.random.default_rng(semilla)
    dD = np.gradient(Dx_tab, x_tab)
    x = x0.copy(); dt = T_min / sub
    salida = [x.copy()]
    for _ in range(n_ondas):
        for _ in range(sub):
            D = np.interp(x, x_tab, Dx_tab); D1 = np.interp(x, x_tab, dD)
            x = x + D1 * dt + np.sqrt(2 * D * dt) * rng.standard_normal(x.size)
            x = np.where(x < 0, -x, x); x = np.where(x > L, 2 * L - x, x)
        salida.append(x.copy())
    return np.array(salida)

### 3.2 Modelo integrado (fase 6) y extensión con ingesta

In [ ]:
@dataclass
class ParamInt:
    # --- geometría y malla del bucle lento ---
    Lx: float = 10.0            # cm   longitud (fondo -> píloro)
    Ly: float = 4.0             # cm   anchura de la sección
    nx: int = 25
    ny: int = 10
    x_sec: float = 6.0          # cm   las paredes secretan para x < x_sec (cuerpo y fondo)
    frac_ini: float = 0.6       # —    la comida sólida empieza en el 60 % proximal
    # --- química (fase 1b, tampón en equilibrio) ---
    k_a1: float = 1.7e-3; k_a2: float = 0.83; pKa: tuple = (2.5, 3.5, 4.5, 5.5, 6.5)
    beta_prot: float = 0.30     # mmol/(g·pH)
    r_acid: float = 0.375       # mmol/min (estimulada)
    q_s: float = 2.5            # mL/min   caudal de jugo (estimulado)
    r_G: float = 1.0e-4         # mmol/min
    f_basal: float = 0.10
    D_H: float = 5.6e-3         # cm²/min difusión molecular del H+
    escala_D: float = 1.0       # factor sobre la dispersión medida con LBM (incertidumbre del promediado)
    # --- vaciado ---
    kcal_min: float = 2.0
    V_res: float = 30.0
    # --- partículas (fase 4) ---
    rho_p: float = 1.05e-3; d50_ini: float = 3.0; sg_ini: float = 0.5
    q_atrapada: float = 0.5; g_min: float = 0.5; F_min: float = 0.2; F_max: float = 2.0
    sigma_c0: float = 1.0e5; w_inf: float = 0.3; X_w: float = 60.0; pH_ref: float = 4.0
    r_e0: float = 5e-3; pH_opt: float = 2.0; sigma_pH: float = 1.5; K_E: float = 0.005
    d_c: float = 1.5
    gamma_pared: float = 0.5    # —    cuánto arrastra la pared que se contrae a las partículas apoyadas (0: nada, 1: todo)
    n_part: int = 4000; n_max: int = 30000


@dataclass
class Comida6:
    nombre: str
    V_comida: float        # mL (sólidos + bebida)
    prot_tampon: float     # g de proteína que aporta tampón
    masa_solida: float     # g de sólido marcado (partículas)
    kcal: float
    pH0: float = 6.0
    V_basal: float = 25.0


def resolver_h_vec(T, Bt, pKa, h0=None, tol=1e-12, max_iter=60):
    n = len(pKa); Ka = [10 ** (3.0 - pk) for pk in pKa]; Bj = Bt / n
    h = np.clip(T if h0 is None else h0, 1e-12, None); h = np.minimum(h, np.maximum(T, 1e-12))
    for _ in range(max_iter):
        F = h - T; dF = np.ones_like(h)
        for K in Ka:
            F = F + Bj * h / (h + K); dF = dF + Bj * K / (h + K) ** 2
        hn = np.where(h - F / dF <= 0, 0.1 * h, h - F / dF)
        if np.max(np.abs(hn - h) / hn) < tol:
            return hn
        h = hn
    return h


def difusion_variable(c, Dx_caras, Dy_celdas, dx, dy):
    # div(D grad c) con D(x) variable y paredes sin flujo. c: (..., ny, nx)
    Fx = np.zeros(c.shape[:-1] + (c.shape[-1] + 1,))
    Fx[..., 1:-1] = -Dx_caras[1:-1] * np.diff(c, axis=-1) / dx
    Fy = np.zeros(c.shape[:-2] + (c.shape[-2] + 1, c.shape[-1]))
    Fy[..., 1:-1, :] = -Dy_celdas * np.diff(c, axis=-2) / dy
    return -(Fx[..., 1:] - Fx[..., :-1]) / dx - (Fy[..., 1:, :] - Fy[..., :-1, :]) / dy


iT, iB, iG, iE, iP = range(5)
NOMBRES_CAMPOS = ["T", "B", "G", "E", "P"]


class ModeloIntegrado:
    # Bucle lento único: química local + transporte + partículas lagrangianas + píloro abierto + volumen variable.
    # El flujo resuelto con LBM (escala rápida) entra a través de mapas promediados en fase:
    # dispersión del líquido D(x), dispersión y frecuencia de molienda de las partículas.
    # Además, el vaciado genera un flujo neto hacia el píloro u_b(x) (el contenido avanza al salir por el píloro).
    def __init__(self, pi, comida, mapas, semilla=0, guardar_cada=1.0):
        self.p, self.c, self.m = pi, comida, mapas
        p = pi
        self.rng = np.random.default_rng(semilla)
        self.guardar_cada = guardar_cada
        self.V0 = comida.V_comida + comida.V_basal
        self.T_e = comida.kcal / p.kcal_min
        self.dx, self.dy = p.Lx / p.nx, p.Ly / p.ny
        self.x = (np.arange(p.nx) + 0.5) * self.dx
        self.xf = np.arange(p.nx + 1) * self.dx
        self.xm = mapas["x"]
        self.Dx = p.escala_D * np.interp(self.x, self.xm, mapas["fluido"]["Dx"])
        self.Dy = p.escala_D * np.interp(self.x, self.xm, mapas["fluido"]["Dy"])
        self.Dx_caras = np.interp(self.xf, self.x, self.Dx)
        self.Dx_p = p.escala_D * mapas["particula"]["Dx"]; self.lam_g = mapas["particula"]["lambda_g"]
        self.mask_sec = np.zeros((p.ny, p.nx)); self.mask_sec[[0, -1], :] = self.x < p.x_sec
        self.n_sec = self.mask_sec.sum()
        # cantidades por celda (mmol para T, B, G, E; g para P)
        V = self.V0; Vcel = self.Vcel(V)
        n = len(p.pKa)
        Bt = p.beta_prot * comida.prot_tampon * (max(p.pKa) - min(p.pKa) + 1) / V * 1000      # mM
        h0 = 10 ** (3 - comida.pH0)
        ligados = sum((Bt / n) / (1 + 10 ** (comida.pH0 - pk)) for pk in p.pKa)
        H_bas = 10 ** (3 - 1.5) * comida.V_basal / V
        conc = np.zeros((5, p.ny, p.nx))
        conc[iT] = ligados + h0 + H_bas; conc[iB] = Bt; conc[iE] = 0.04 * comida.V_basal / V
        self.N = conc * Vcel / 1000
        self.V = V
        self.h = resolver_h_vec(conc[iT], conc[iB], p.pKa)
        # partículas (lognormal en masa) en la parte proximal
        d = np.exp(self.rng.normal(np.log(p.d50_ini) - 3 * p.sg_ini ** 2, p.sg_ini, p.n_part))
        self.d = d; self.w = np.full(d.size, comida.masa_solida / self.masa(d).sum())
        self.xp = self.rng.uniform(0, p.frac_ini * p.Lx, d.size); self.X = np.zeros(d.size)
        self.sale = {"T": 0.0, "B": 0.0, "G": 0.0, "E": 0.0, "P": 0.0, "solido": 0.0, "liquido": 0.0}
        self.secretado = {"T": 0.0, "G": 0.0}
        self.t = 0.0
        self.hist = {k: [] for k in ("t", "V", "solido_g", "P_g", "pH_medio", "pH_min", "pH_max", "pH_particulas",
                                     "d50", "n_part", "salida_solida_g", "salida_disuelta_g", "u_piloro")}
        self.instantaneas = []

    # ---------- utilidades ----------
    def masa(self, d):
        return self.p.rho_p * np.pi * d ** 3 / 6

    def Vcel(self, V):
        return V / (self.p.nx * self.p.ny)                     # mL por celda (la losa se adelgaza uniformemente)

    def concentraciones(self, N=None, V=None):
        N = self.N if N is None else N
        return N / self.Vcel(self.V if V is None else V) * 1000

    def caudales(self, t):
        p = self.p
        if t < self.T_e:
            return p.q_s, p.q_s + (self.V0 - p.V_res) / self.T_e, 1.0
        return p.f_basal * p.q_s, p.f_basal * p.q_s, p.f_basal

    def velocidad_neta(self, V, q_in, Q):
        # u_b en las caras x (cm/min) a partir del balance de volumen por columnas:
        # lo que entra por la pared (jugo) menos lo que pierde la columna al adelgazarse se va hacia el píloro.
        p = self.p
        W = V / (p.Lx * p.Ly)
        dVdt = q_in - Q
        sec_col = self.mask_sec.sum(axis=0) / max(self.n_sec, 1) * q_in            # mL/min que entran en cada columna
        perdida_col = dVdt / p.nx                                                    # mL/min (uniforme)
        caudal_caras = np.concatenate([[0.0], np.cumsum(sec_col - perdida_col)])     # mL/min a través de cada cara
        return caudal_caras / (p.Ly * W), caudal_caras

    def flujo_transversal(self, V, q_in, Q):
        p = self.p
        _, caudal = self.velocidad_neta(V, q_in, Q)
        sec = self.mask_sec / max(self.n_sec, 1) * q_in                              # mL/min que entran en cada celda
        axial = (caudal[1:] - caudal[:-1]) / p.ny                                    # mL/min que salen de cada fila hacia x+
        encoge = (q_in - Q) / (p.nx * p.ny)                                          # mL/min de cambio de volumen por celda
        neto = sec - axial[None, :] - encoge                                         # (ny, nx): lo que debe salir en vertical
        G = np.zeros((p.ny + 1, p.nx))
        G[1:] = np.cumsum(neto, axis=0)                                              # flujo hacia arriba por cada cara
        G[-1] = 0.0                                                                  # (es cero por construcción)
        return G

    def rhs(self, N, h, V, q_in, Q, f_sec):
        p = self.p
        Vc = self.Vcel(V)
        c = N / Vc * 1000
        dN = difusion_variable(c, self.Dx_caras, self.Dy, self.dx, self.dy) * Vc / 1000
        dN[iT] += difusion_variable(h, np.full(p.nx + 1, p.D_H), np.full(p.nx, p.D_H), self.dx, self.dy) * Vc / 1000
        # advección por el flujo neto hacia el píloro (upwind; u_b >= 0). La última cara es la salida pilórica.
        u, caudal = self.velocidad_neta(V, q_in, Q)
        flujo = np.zeros(c.shape[:-1] + (p.nx + 1,))
        # upwind según el signo del caudal en cada cara (durante la ingesta puede ir hacia el fundus)
        c_cara = np.where(caudal[1:-1] > 0, c[..., :-1], c[..., 1:])
        flujo[..., 1:-1] = (caudal[1:-1] / p.ny) * c_cara / 1000                   # mmol/min (o g/min) por fila
        flujo[..., -1] = (max(caudal[-1], 0.0) / p.ny) * c[..., -1] / 1000          # salida pilórica
        dN += flujo[..., :-1] - flujo[..., 1:]
        # flujo transversal: el jugo entra por las filas de pared y empuja el contenido hacia el centro
        # (balance de volumen fila a fila; así el jugo se diluye y nada supera su concentración)
        G = self.flujo_transversal(V, q_in, Q)                                       # (ny+1, nx) mL/min
        cu = np.where(G[1:-1] > 0, c[..., :-1, :], c[..., 1:, :])                    # upwind
        Fy = np.zeros(c.shape[:-2] + (p.ny + 1, p.nx))
        Fy[..., 1:-1, :] = G[1:-1] * cu / 1000
        dN += Fy[..., :-1, :] - Fy[..., 1:, :]
        # química: activación del pepsinógeno
        r_a = (p.k_a1 * h + p.k_a2 * h * c[iE]) * c[iG] * Vc / 1000
        dN[iG] -= r_a; dN[iE] += r_a
        # secreción en las paredes
        dN[iT] += self.mask_sec * f_sec * p.r_acid / self.n_sec
        dN[iG] += self.mask_sec * f_sec * p.r_G / self.n_sec
        salida = flujo[..., -1].sum(axis=-1)                                        # por especie
        return dN, salida

    def paso_campos(self, dt):
        p = self.p
        q_in, Q, f_sec = self.caudales(self.t)
        V0 = self.V; V1 = V0 + (q_in - Q) * dt
        N0, h0 = self.N, self.h
        k1, s1 = self.rhs(N0, h0, V0, q_in, Q, f_sec)
        N1 = N0 + dt * k1
        c1 = N1 / self.Vcel(V1) * 1000
        h1 = resolver_h_vec(c1[iT], c1[iB], p.pKa, h0)
        k2, s2 = self.rhs(N1, h1, V1, q_in, Q, f_sec)
        self.N = N0 + 0.5 * dt * (k1 + k2)
        sal = 0.5 * dt * (s1 + s2)
        for k, i in zip(NOMBRES_CAMPOS, range(5)):
            self.sale[k] += float(sal[i])
        self.secretado["T"] += f_sec * p.r_acid * dt; self.secretado["G"] += f_sec * p.r_G * dt
        self.sale["liquido"] += Q * dt
        self.V = V1
        c = self.concentraciones()
        self.h = resolver_h_vec(c[iT], c[iB], p.pKa, h1)
        return q_in, Q

    def paso(self, dt):
        Dmax = max(self.Dx.max(), self.Dy.max()) + self.p.D_H
        dt_dif = 0.35 / (Dmax * (1 / self.dx ** 2 + 1 / self.dy ** 2))
        q_in, Q, _ = self.caudales(self.t)
        u, _ = self.velocidad_neta(self.V, q_in, Q)
        dt_adv = 0.5 * self.dx / max(np.abs(u).max(), 1e-9)
        G = self.flujo_transversal(self.V, q_in, Q)
        dt_adv = min(dt_adv, 0.5 * self.Vcel(self.V) / max(np.abs(G).max(), 1e-9))
        n_sub = int(math.ceil(dt / min(dt_dif, dt_adv)))
        for _ in range(n_sub):
            q_in, Q = self.paso_campos(dt / n_sub)
        self.paso_particulas(dt, q_in, Q)
        self.t += dt

    def paso_particulas(self, dt, q_in, Q):
        p, rng = self.p, self.rng
        if self.d.size == 0:
            return
        # transporte: flujo neto hacia el píloro + dispersión axial medida con LBM (Itô, D variable)
        u, _ = self.velocidad_neta(self.V, q_in, Q)
        # las partículas sedimentadas avanzan con la velocidad del líquido a la altura de su centro (perfil parabólico)
        yc = np.minimum(0.5 * self.d / 10, 0.5 * p.Ly)                             # cm
        alfa = 6 * (yc / p.Ly) * (1 - yc / p.Ly)
        ub = alfa * np.interp(self.xp, self.xf, u)
        # además, apoyadas en la pared, se mueven con ella: el estómago se contrae (tono) al vaciarse,
        # con L ∝ V^(1/3), y el píloro es el punto fijo -> deriva hacia el píloro proporcional a la distancia
        dlnL = (q_in - Q) / (3 * self.V)
        ub = ub - p.gamma_pared * (p.Lx - self.xp) * dlnL
        D = np.interp(self.xp, self.xm, self.Dx_p); dD = np.interp(self.xp, self.xm, np.gradient(self.Dx_p, self.xm))
        xn = self.xp + (ub + dD) * dt + np.sqrt(2 * D * dt) * rng.standard_normal(self.xp.size)
        xn = np.abs(xn)                                                     # fondo: reflexión
        cruza = xn >= p.Lx
        pasa = cruza & (self.d < p.d_c)                                     # criba pilórica
        self.sale["solido"] += float(np.sum(self.w[pasa] * self.masa(self.d[pasa])))
        xn = np.where(cruza & ~pasa, 2 * p.Lx - xn - 1e-9, xn)              # las grandes vuelven (retropulsión)
        quedan = ~pasa
        for nombre, val in (("xp", xn), ("d", self.d), ("w", self.w), ("X", self.X)):
            setattr(self, nombre, val[quedan])
        if self.d.size == 0:
            return
        i = np.minimum((self.xp / self.dx).astype(int), p.nx - 1)
        c = self.concentraciones()
        pH = 3 - np.log10(self.h[0, i]); E = c[iE][0, i]                   # viven en la capa del fondo
        # ablandamiento por el ácido LOCAL y rotura en la zona de molienda
        self.X += np.maximum(0.0, p.pH_ref - pH) * dt
        sig = p.sigma_c0 * (p.w_inf + (1 - p.w_inf) * np.exp(-self.X / p.X_w))
        Prot = np.clip((p.F_max - sig * (self.d * 1e-3) ** 2) / (p.F_max - p.F_min), 0, 1)
        # en cada contracción terminal (frecuencia λ_g) la partícula atrapada se rompe si es grande
        # o, si cabe por el píloro, sale expulsada con el pulso transpilórico (criba antral)
        lam = np.interp(self.xp, self.xm, self.lam_g) * p.q_atrapada
        expulsa = (self.d < p.d_c) & (rng.random(self.d.size) < 1 - np.exp(-lam * dt))
        self.sale["solido"] += float(np.sum(self.w[expulsa] * self.masa(self.d[expulsa])))
        for nombre in ("xp", "d", "w", "X"):
            setattr(self, nombre, getattr(self, nombre)[~expulsa])
        i, pH, E, sig = i[~expulsa], pH[~expulsa], E[~expulsa], sig[~expulsa]
        Prot = np.clip((p.F_max - sig * (self.d * 1e-3) ** 2) / (p.F_max - p.F_min), 0, 1)
        S = np.interp(self.xp, self.xm, self.lam_g) * p.q_atrapada * (self.d > p.g_min) * Prot
        rompe = rng.random(self.d.size) < 1 - np.exp(-S * dt)
        fac = 2 ** (-1 / 3)
        self.d = np.concatenate([np.where(rompe, self.d * fac, self.d), self.d[rompe] * fac])
        for nombre in ("w", "xp", "X"):
            setattr(self, nombre, np.concatenate([getattr(self, nombre), getattr(self, nombre)[rompe]]))
        i = np.concatenate([i, i[rompe]]); pH = np.concatenate([pH, pH[rompe]]); E = np.concatenate([E, E[rompe]])
        # erosión con pH y pepsina LOCALES; lo erosionado se convierte en péptidos disueltos en esa celda
        r = p.r_e0 * np.exp(-((pH - p.pH_opt) / p.sigma_pH) ** 2) * E / (E + p.K_E)
        d_n = np.maximum(self.d - r * dt, 0.0)
        dm = self.w * (self.masa(self.d) - self.masa(d_n))
        np.add.at(self.N[iP][0], i, dm)
        vivo = d_n > 0
        self.d = d_n
        for nombre in ("d", "w", "xp", "X"):
            setattr(self, nombre, getattr(self, nombre)[vivo])
        if self.d.size > p.n_max:                                             # aclarado estratificado (fase 4)
            clase = np.floor(3 * np.log2(self.d)).astype(int)
            _, inv, cuenta = np.unique(clase, return_inverse=True, return_counts=True)
            poblada = cuenta[inv] > 200
            quita = poblada & (rng.random(self.d.size) < 0.5)
            self.w = np.where(poblada, 2 * self.w, self.w)
            for nombre in ("d", "w", "xp", "X"):
                setattr(self, nombre, getattr(self, nombre)[~quita])

    # ---------- registro ----------
    def totales(self):
        tot = {k: float(self.N[i].sum()) for k, i in zip(NOMBRES_CAMPOS, range(5))}
        tot["solido"] = float(np.sum(self.w * self.masa(self.d)))
        return tot

    def registrar(self, instantanea=False):
        H = self.hist; tot = self.totales()
        pHc = 3 - np.log10(self.h)
        H["t"].append(self.t); H["V"].append(self.V)
        H["solido_g"].append(tot["solido"]); H["P_g"].append(tot["P"])
        H["pH_medio"].append(float(3 - np.log10(np.mean(self.h))))
        H["pH_min"].append(float(pHc.min())); H["pH_max"].append(float(pHc.max()))
        if self.d.size:
            i = np.minimum((self.xp / self.dx).astype(int), self.p.nx - 1)
            ms = self.w * self.masa(self.d); o = np.argsort(self.d)
            H["pH_particulas"].append(float(np.sum(ms * pHc[0, i]) / ms.sum()))
            H["d50"].append(float(np.interp(0.5, np.cumsum(ms[o]) / ms.sum(), self.d[o])))
        else:
            H["pH_particulas"].append(np.nan); H["d50"].append(np.nan)
        H["n_part"].append(int(self.d.size))
        H["salida_solida_g"].append(self.sale["solido"]); H["salida_disuelta_g"].append(self.sale["P"])
        q_in, Q, _ = self.caudales(self.t)
        H["u_piloro"].append(float(self.velocidad_neta(self.V, q_in, Q)[0][-1]))
        if instantanea:
            self.instantaneas.append({"t": self.t, "pH": pHc.copy(), "P": self.concentraciones()[iP].copy(), "V": self.V,
                                      "xp": self.xp.copy(), "d": self.d.copy(), "w": self.w.copy()})

    def simular(self, t_final, dt, cada_inst=None, informar=True):
        n = int(round(t_final / dt)); cada = max(1, int(round(self.guardar_cada / dt)))
        cada_i = None if cada_inst is None else max(1, int(round(cada_inst / dt)))
        t0 = time.time()
        for k in range(n + 1):
            if k % cada == 0:
                self.registrar(instantanea=(cada_i is not None and k % cada_i == 0))
            if k == n:
                break
            self.paso(dt)
            if informar and (k + 1) % max(1, n // 4) == 0:
                print(f"  t = {self.t:5.0f} min  ({time.time() - t0:4.0f} s)")
        self.hist = {k: np.array(v) for k, v in self.hist.items()}
        return self

    def retencion(self):
        H = self.hist
        return 100 * (H["solido_g"] + H["P_g"]) / self.c.masa_solida

In [ ]:
iM = 5                                       # nuevo campo: fracción de comida en el líquido (trazador)
NOMBRES6 = ["T", "B", "G", "E", "P", "M"]


@dataclass
class Ingesta:
    duracion: float = 6.0        # min   la comida se toma en 6 min
    tragos: int = 18             # nº de tragos (uno cada 20 s)
    x_cardias: tuple = (0.4, 1.6)   # cm   tramo de la curvatura menor por donde entra (fila superior del modelo)


class ModeloConIngesta(ModeloIntegrado):
    # El modelo integrado de la fase 6 con tres añadidos para la representación:
    #  1) INGESTA: el estómago empieza en ayunas (25 mL de jugo) y la comida entra por el cardias en tragos.
    #  2) TRAZADOR de comida M: fracción del líquido que procede de la comida (1 = comida pura, 0 = jugo).
    #     Muestra la dilución progresiva por el jugo y su salida por el píloro.
    #  3) Registro de las partículas que salen (para dibujarlas en el duodeno).
    def __init__(self, pi, comida, mapas, ingesta=None, semilla=0):
        super().__init__(pi, comida, mapas, semilla=semilla)
        p = pi
        self.ing = ingesta or Ingesta()
        # estado inicial: estómago en ayunas
        V = comida.V_basal; Vcel = self.Vcel(V)
        conc = np.zeros((6, p.ny, p.nx))
        conc[iT] = 10 ** (3 - 1.5); conc[iE] = 0.04
        self.N = conc * Vcel / 1000; self.V = V
        self.h = resolver_h_vec(conc[iT], np.zeros_like(conc[iT]) + 1e-9, p.pKa)
        # composición del líquido de la comida (mM): tampón y protones a pH de la comida
        n = len(p.pKa)
        self.B_comida = p.beta_prot * comida.prot_tampon * (max(p.pKa) - min(p.pKa) + 1) / comida.V_comida * 1000
        h0 = 10 ** (3 - comida.pH0)
        self.T_comida = h0 + sum((self.B_comida / n) / (1 + 10 ** (comida.pH0 - pk)) for pk in p.pKa)
        self.mask_cardias = np.zeros((p.ny, p.nx))
        self.mask_cardias[-1, (self.x > self.ing.x_cardias[0]) & (self.x < self.ing.x_cardias[1])] = 1.0
        self.n_card = self.mask_cardias.sum()
        # las partículas esperan su trago: cada una entra en un instante discreto
        trago = self.rng.integers(0, self.ing.tragos, self.d.size)
        self.t_entrada_cola = (trago + 0.5) * self.ing.duracion / self.ing.tragos
        self.cola = {"d": self.d, "w": self.w}
        self.d = np.empty(0); self.w = np.empty(0); self.xp = np.empty(0); self.X = np.empty(0)
        self.salidas = []                                  # (t, d mediano, masa en g) de lo que pasa el píloro
        self.sale["M"] = 0.0; self.entra = {"M": 0.0, "B": 0.0, "T": 0.0, "volumen": 0.0}
        self.V_max = comida.V_comida + comida.V_basal
        self.hist["frac_comida_media"] = []; self.hist["en_cola"] = []

    # ---------- caudales y fuentes de volumen ----------
    def q_comida(self, t):
        return self.c.V_comida / self.ing.duracion if t < self.ing.duracion else 0.0

    def caudales(self, t):
        p = self.p
        qm = self.q_comida(t)
        if t < self.T_e:
            Q = p.q_s + (self.V_max - p.V_res) / self.T_e
            return p.q_s + qm, Q, 1.0
        return p.f_basal * p.q_s + qm, p.f_basal * p.q_s, p.f_basal

    def fuentes_volumen(self, t):
        # mL/min que entran en cada celda: jugo por las paredes secretoras, comida por el cardias
        q_in, Q, f_sec = self.caudales(t)
        jugo = self.mask_sec / max(self.n_sec, 1) * f_sec * self.p.q_s
        comida = self.mask_cardias / max(self.n_card, 1) * self.q_comida(t)
        return jugo + comida

    def velocidad_neta(self, V, q_in, Q):
        p = self.p
        W = V / (p.Lx * p.Ly)
        fuente_col = self.fuentes_volumen(self.t).sum(axis=0)
        perdida_col = (q_in - Q) / p.nx
        caudal_caras = np.concatenate([[0.0], np.cumsum(fuente_col - perdida_col)])
        return caudal_caras / (p.Ly * W), caudal_caras

    def flujo_transversal(self, V, q_in, Q):
        p = self.p
        _, caudal = self.velocidad_neta(V, q_in, Q)
        axial = (caudal[1:] - caudal[:-1]) / p.ny
        encoge = (q_in - Q) / (p.nx * p.ny)
        neto = self.fuentes_volumen(self.t) - axial[None, :] - encoge
        G = np.zeros((p.ny + 1, p.nx)); G[1:] = np.cumsum(neto, axis=0); G[-1] = 0.0
        return G

    def rhs(self, N, h, V, q_in, Q, f_sec):
        dN, salida = super().rhs(N, h, V, q_in, Q, f_sec)      # transporte, química, secreción y salida
        qm = self.q_comida(self.t)
        if qm > 0:
            reparto = self.mask_cardias / self.n_card * qm / 1000          # L/min por celda del cardias
            dN[iT] += reparto * self.T_comida
            dN[iB] += reparto * self.B_comida
            dN[iM] += reparto * 1000.0                                      # trazador: 1000 "mM" = comida pura
        return dN, salida

    def paso_campos(self, dt):
        p = self.p
        q_in, Q, f_sec = self.caudales(self.t)
        V0 = self.V; V1 = V0 + (q_in - Q) * dt
        N0, h0 = self.N, self.h
        k1, s1 = self.rhs(N0, h0, V0, q_in, Q, f_sec)
        N1 = N0 + dt * k1
        c1 = N1 / self.Vcel(V1) * 1000
        h1 = resolver_h_vec(c1[iT], np.maximum(c1[iB], 1e-9), p.pKa, h0)
        k2, s2 = self.rhs(N1, h1, V1, q_in, Q, f_sec)
        self.N = N0 + 0.5 * dt * (k1 + k2)
        sal = 0.5 * dt * (s1 + s2)
        for k, i in zip(NOMBRES6, range(6)):
            self.sale[k] = self.sale.get(k, 0.0) + float(sal[i])
        qm = self.q_comida(self.t)
        self.entra["M"] += qm * dt; self.entra["volumen"] += qm * dt
        self.entra["B"] += qm / 1000 * self.B_comida * dt; self.entra["T"] += qm / 1000 * self.T_comida * dt
        self.secretado["T"] += f_sec * p.r_acid * dt; self.secretado["G"] += f_sec * p.r_G * dt
        self.sale["liquido"] += Q * dt
        self.V = V1
        c = self.concentraciones()
        self.h = resolver_h_vec(c[iT], np.maximum(c[iB], 1e-9), p.pKa, h1)
        return q_in, Q

    def paso_particulas(self, dt, q_in, Q):
        # 1) entran las partículas de los tragos que tocan
        if self.cola["d"].size:
            ya = self.t_entrada_cola <= self.t + dt
            if ya.any():
                n = int(ya.sum())
                x_in = self.rng.uniform(*self.ing.x_cardias, n)
                self.d = np.concatenate([self.d, self.cola["d"][ya]]); self.w = np.concatenate([self.w, self.cola["w"][ya]])
                self.xp = np.concatenate([self.xp, x_in]); self.X = np.concatenate([self.X, np.zeros(n)])
                self.cola = {"d": self.cola["d"][~ya], "w": self.cola["w"][~ya]}
                self.t_entrada_cola = self.t_entrada_cola[~ya]
        # 2) dinámica de la fase 6; se anotan las que salen comparando antes y después
        antes = self.sale["solido"]
        d_ant = self.d.copy()
        super().paso_particulas(dt, q_in, Q)
        if self.sale["solido"] > antes:
            # se registra una muestra representativa de lo que sale (tamaño medio de las candidatas a salir)
            cand = d_ant[d_ant < self.p.d_c]
            if cand.size:
                self.salidas.append((self.t, float(np.median(cand)), self.sale["solido"] - antes))

    # ---------- registro ----------
    def totales(self):
        tot = {k: float(self.N[i].sum()) for k, i in zip(NOMBRES6, range(6))}
        tot["solido"] = float(np.sum(self.w * self.masa(self.d))) if self.d.size else 0.0
        tot["cola"] = float(np.sum(self.cola["w"] * self.masa(self.cola["d"]))) if self.cola["d"].size else 0.0
        return tot

    def instantanea(self):
        c = self.concentraciones()
        return {"t": self.t, "V": self.V, "pH": 3 - np.log10(self.h), "M": c[iM] / 1000, "P": c[iP].copy(),
                "xp": self.xp.copy(), "d": self.d.copy(), "w": self.w.copy(), "X": self.X.copy(),
                "salidas": [e for e in self.salidas if self.t - e[0] < 10.0]}    # solo las recientes (para el duodeno)

    def simular_con_instantaneas(self, tiempos, dt, informar=True):
        # avanza hasta cada tiempo pedido y guarda una instantánea; además, historia cada minuto
        tiempos = np.sort(np.asarray(tiempos, float))
        fotos, t0 = [], time.time()
        prox_hist = 0.0
        for k, tf in enumerate(tiempos):
            while self.t < tf - 1e-9:
                paso = min(dt, tf - self.t)
                if self.t >= prox_hist - 1e-9:
                    self.registrar(); self.hist["frac_comida_media"].append(float(np.mean(self.concentraciones()[iM]) / 1000))
                    self.hist["en_cola"].append(self.totales()["cola"]); prox_hist += 1.0
                self.paso(paso)
            fotos.append(self.instantanea())
            if informar and (k + 1) % max(1, len(tiempos) // 4) == 0:
                print(f"  t = {self.t:6.1f} min  ({time.time() - t0:4.0f} s)")
        self.hist = {k: np.array(v) for k, v in self.hist.items()}
        return fotos

### 3.3 Anatomía y dibujo

In [ ]:
class Anatomia:
    # Dibujo esquemático de un estómago en "J" sobre el que se proyecta la losa del modelo (x ∈ [0, Lx], y ∈ [0, Ly]).
    #   · x (fondo -> píloro) se convierte en la longitud de arco de una línea media curva.
    #   · y = 0 es la CURVATURA MAYOR (donde se apoyan las partículas) e y = Ly la CURVATURA MENOR (cardias).
    #   · La anchura depende de x (fundus ancho, antro estrecho), del volumen (el estómago se distiende al comer)
    #     y de las ondas peristálticas, que estrechan el antro mientras viajan hacia el píloro.
    # Es una REPRESENTACIÓN: los valores son los del modelo 2D, pero la forma y la deformación son un dibujo.
    PUNTOS = np.array([(3.0, 10.4), (2.55, 8.4), (2.65, 6.3), (3.35, 4.3), (4.75, 2.85),
                       (6.5, 2.55), (8.0, 3.25), (9.1, 4.35)])
    X_ANCHO = np.array([0.0, 1.5, 3.0, 4.5, 6.0, 7.5, 8.8, 9.6, 10.0])
    ANCHO = np.array([4.2, 4.4, 3.9, 3.3, 2.9, 2.5, 1.9, 1.25, 1.0])     # anchura con el estómago lleno (unidades del dibujo)

    def __init__(self, Lx=10.0, Ly=4.0, V_max=340.0, onda=None, cap=1.6):
        from scipy.interpolate import CubicSpline
        self.Lx, self.Ly, self.V_max, self.cap = Lx, Ly, V_max, cap
        self.onda = onda or dict(x_ini=4.5, x_plena=6.5, c=0.25, T=20.0, oclusion=0.6, sigma=0.55)
        P = self.PUNTOS
        cuerda = np.r_[0, np.cumsum(np.hypot(*np.diff(P, axis=0).T))]
        sp = CubicSpline(cuerda, P, axis=0)
        u = np.linspace(0, cuerda[-1], 3000)
        C = sp(u); dC = sp(u, 1)
        arco = np.r_[0, np.cumsum(np.hypot(*np.diff(C, axis=0).T))]
        self._s = arco / arco[-1] * Lx                       # x del modelo para cada punto de la línea media
        self._C = C
        tg = dC / np.hypot(dC[:, 0], dC[:, 1])[:, None]
        self._t = tg
        self._n = np.c_[-tg[:, 1], tg[:, 0]]                  # normal izquierda = hacia la curvatura menor

    # ---------- geometría ----------
    def linea_media(self, x):
        x = np.clip(x, 0, self.Lx)
        cx = np.interp(x, self._s, self._C[:, 0]); cy = np.interp(x, self._s, self._C[:, 1])
        nx = np.interp(x, self._s, self._n[:, 0]); ny = np.interp(x, self._s, self._n[:, 1])
        tx = np.interp(x, self._s, self._t[:, 0]); ty = np.interp(x, self._s, self._t[:, 1])
        return np.stack([cx, cy], -1), np.stack([nx, ny], -1), np.stack([tx, ty], -1)

    def oclusion(self, x, tau):
        # ondas gaussianas que nacen en x_ini, alcanzan amplitud plena en x_plena y viajan a c (cm/s) cada T s
        o = self.onda
        rampa = np.clip((x - o["x_ini"]) / (o["x_plena"] - o["x_ini"]), 0, 1)
        lam = o["c"] * o["T"]
        fase = (tau % o["T"]) * o["c"]                                      # posición de la onda más reciente
        occ = np.zeros_like(x)
        for k in range(int(math.ceil((self.Lx - o["x_ini"]) / lam)) + 2):
            xk = o["x_ini"] + fase + k * lam
            occ = occ + np.exp(-((x - xk) / o["sigma"]) ** 2)
        return o["oclusion"] * rampa * np.minimum(occ, 1.0)

    def anchura(self, x, V, tau):
        vf = np.clip(V / self.V_max, 0, 1.2)
        f_prox = 0.35 + 0.65 * vf ** 0.6                      # el fundus y el cuerpo se relajan para alojar la comida
        f_ant = 0.70 + 0.30 * vf ** 0.6                       # el antro cambia poco de calibre
        mezcla = np.clip((x - 4.0) / 3.0, 0, 1)
        f = (1 - mezcla) * f_prox + mezcla * f_ant
        return np.interp(np.clip(x, 0, self.Lx), self.X_ANCHO, self.ANCHO) * f * (1 - self.oclusion(x, tau))

    def punto(self, x, y, V, tau):
        # (x, y) del modelo -> coordenadas del dibujo. x < 0: casquete del fundus (cúpula que cierra el extremo)
        x = np.asarray(x, float); y = np.asarray(y, float)
        C, n, t = self.linea_media(x)
        w = self.anchura(np.maximum(x, 0), V, tau)
        u = np.clip(-x / self.cap, 0, 1)                                        # 0 en x = 0, 1 en la punta de la cúpula
        w = np.where(x < 0, self.anchura(np.zeros_like(x), V, tau) * np.sqrt(np.clip(1 - u ** 2, 0, 1)), w)
        vf = 0.35 + 0.65 * np.clip(V / self.V_max, 0, 1.2) ** 0.6
        C = np.where((x < 0)[..., None], C - t * (u * self.cap * vf)[..., None], C)
        return C + (y / self.Ly - 0.5)[..., None] * w[..., None] * n

    def malla(self, V, tau, nxf=110, nyf=26, ncap=14):
        xs = np.r_[np.linspace(-self.cap, 0, ncap, endpoint=False), np.linspace(0, self.Lx, nxf)]
        ys = np.linspace(0, self.Ly, nyf)
        XX, YY = np.meshgrid(xs, ys)
        P = self.punto(XX, YY, V, tau)
        return P[..., 0], P[..., 1], xs, ys

    def contorno(self, V, tau, n=300):
        xs = np.r_[np.linspace(-self.cap, 0, 40, endpoint=False), np.linspace(0, self.Lx, n)]
        mayor = self.punto(xs, np.zeros_like(xs), V, tau)
        menor = self.punto(xs, np.full_like(xs, self.Ly), V, tau)
        return mayor, menor

    # ---------- tubos de entrada y salida ----------
    def esofago(self, V, tau, x_c=1.0, largo=3.2, ancho=0.9, n=40):
        B = self.punto(np.array(x_c), np.array(self.Ly), V, tau)
        _, nrm, tg = self.linea_media(np.array(x_c))
        d0 = 0.75 * nrm + 0.25 * np.array([0.0, 1.0]); d0 /= np.hypot(*d0)
        s = np.linspace(0, 1, n)[:, None]
        fin = B + largo * np.array([0.45, 0.9])
        ctrl = B + 1.2 * d0
        L = (1 - s) ** 2 * B + 2 * s * (1 - s) * ctrl + s ** 2 * fin            # Bézier cuadrática
        return self._tubo(L, ancho), L

    def duodeno(self, V, tau, largo=3.0, ancho=0.85, n=40):
        C, _, tg = self.linea_media(np.array(self.Lx))
        s = np.linspace(0, 1, n)[:, None]
        ctrl = C + 1.4 * tg
        fin = C + 1.4 * tg + np.array([0.6, -largo])
        L = (1 - s) ** 2 * C + 2 * s * (1 - s) * ctrl + s ** 2 * fin
        return self._tubo(L, ancho), L

    @staticmethod
    def _tubo(L, ancho):
        d = np.gradient(L, axis=0); d /= np.hypot(d[:, 0], d[:, 1])[:, None]
        nrm = np.c_[-d[:, 1], d[:, 0]]
        return np.r_[L + 0.5 * ancho * nrm, (L - 0.5 * ancho * nrm)[::-1]]


def remuestrear(C, xs, ys, Lx, Ly):
    # interpolación bilineal de un campo de celdas (ny, nx) a los puntos (ys, xs); fuera de los centros se extiende el borde
    ny, nx = C.shape
    fx = np.clip(np.asarray(xs) / (Lx / nx) - 0.5, 0, nx - 1); fy = np.clip(np.asarray(ys) / (Ly / ny) - 0.5, 0, ny - 1)
    i0 = np.minimum(fx.astype(int), nx - 2); j0 = np.minimum(fy.astype(int), ny - 2)
    ax = fx - i0; ay = fy - j0
    J0, I0 = np.meshgrid(j0, i0, indexing="ij"); AY, AX = np.meshgrid(ay, ax, indexing="ij")
    return ((1 - AX) * (1 - AY) * C[J0, I0] + AX * (1 - AY) * C[J0, I0 + 1]
            + (1 - AX) * AY * C[J0 + 1, I0] + AX * AY * C[J0 + 1, I0 + 1])


def pseudoaleatorio(a, b):
    # número en [0, 1) que depende solo de (a, b): da a cada partícula una altura estable entre fotogramas
    return (np.sin(a * 12.9898 + b * 78.233) * 43758.5453) % 1.0

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Polygon, Circle
from matplotlib import patheffects

PARED = "#b5656f"; PARED_CLARA = "#f3dfe1"; TUBO = "#ead2d5"
CMAP_M = LinearSegmentedColormap.from_list("comida", ["#d8e8f4", "#bcd6c8", "#e9c46a", "#c98a3c", "#7a4a1f"])
CMAP_PH = plt.get_cmap("Spectral")
NORM_M = Normalize(0, 1); NORM_PH = Normalize(0.8, 7.0)
CMAP_D = plt.get_cmap("plasma"); NORM_D = Normalize(0, 4)
PASO_ONDA_S = 5.0          # segundos de "reloj de ondas" por fotograma (en la ingesta coincide con el tiempo real)


def reloj_ondas(fotos):
    return np.arange(len(fotos)) * PASO_ONDA_S


def dibujar_estomago(ax, A, foto, tau, que="M", ingesta=None, rotulos=False, rng_sem=0):
    # Dibuja un fotograma completo de un panel: estómago deformado, campo, partículas, esófago y duodeno.
    ax.clear(); ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-0.2, 11.6); ax.set_ylim(0.6, 14.6)
    V = foto["V"]
    campo, cmap, norm = (foto["M"], CMAP_M, NORM_M) if que == "M" else (foto["pH"], CMAP_PH, NORM_PH)
    # --- esófago y bolos que bajan durante la ingesta ---
    poli_e, eje_e = A.esofago(V, tau)
    ax.add_patch(Polygon(poli_e, closed=True, fc=TUBO, ec=PARED, lw=1.6, zorder=1))
    if ingesta is not None and foto["t"] < ingesta.duracion + 0.05:
        dur_trago = ingesta.duracion / ingesta.tragos
        for k in range(ingesta.tragos):
            t_k = (k + 0.5) * dur_trago                                  # el bolo k llega al cardias en t_k
            f = 1 - (t_k - foto["t"]) / (0.6 * dur_trago)                # recorre el esófago en 0,6 tragos (~12 s)
            if 0 <= f <= 1:
                p = eje_e[int((1 - f) * (len(eje_e) - 1))]
                ax.add_patch(Circle(p, 0.42, fc=cmap(norm(1.0 if que == "M" else 6.0)), ec=PARED, lw=1.0, zorder=2))
    # --- duodeno: lo que sale por el píloro ---
    poli_d, eje_d = A.duodeno(V, tau)
    salida = campo[:, -1].mean()
    ax.add_patch(Polygon(poli_d, closed=True, fc=cmap(norm(salida)) if foto["t"] > 0 else TUBO, ec=PARED, lw=1.6, zorder=1))
    rec = [e for e in foto["salidas"] if foto["t"] - e[0] < 8.0]
    if rec:
        rec = rec[:: max(1, len(rec) // 40)]
        edad = np.array([foto["t"] - e[0] for e in rec]); dd = np.array([e[1] for e in rec])
        idx = np.clip((edad / 8.0 * (len(eje_d) - 1)).astype(int), 0, len(eje_d) - 1)
        lateral = (pseudoaleatorio(np.array([e[0] for e in rec]), dd) - 0.5) * 0.5
        dl = np.gradient(eje_d, axis=0)[idx]; dl /= np.hypot(dl[:, 0], dl[:, 1])[:, None]
        pts = eje_d[idx] + lateral[:, None] * np.c_[-dl[:, 1], dl[:, 0]]
        ax.scatter(pts[:, 0], pts[:, 1], s=6 + 10 * dd, c=dd, cmap=CMAP_D, norm=NORM_D, edgecolors="white", linewidths=0.3, zorder=4)
    # --- el estómago: campo del modelo proyectado sobre la forma deformada ---
    X, Y, xs, ys = A.malla(V, tau)
    C = remuestrear(campo, np.clip(xs, 0, A.Lx), ys, A.Lx, A.Ly)
    ax.pcolormesh(X, Y, C, shading="gouraud", cmap=cmap, norm=norm, zorder=2, rasterized=True)
    mayor, menor = A.contorno(V, tau)
    ax.plot(*np.r_[mayor, menor[::-1], mayor[:1]].T, color=PARED, lw=3.2, zorder=3, solid_joinstyle="round")
    # --- partículas apoyadas en la curvatura mayor (y pequeño); la altura depende del tamaño y de un desorden estable ---
    if foto["d"].size:
        xp, d = foto["xp"], foto["d"]
        # montón: donde hay más partículas la capa es más alta (sedimentan sobre la curvatura mayor)
        bordes = np.linspace(0, A.Lx, 41)
        cnt = np.histogram(xp, bordes)[0].astype(float)
        alto = 0.05 + 0.45 * np.sqrt(cnt / max(cnt.max(), 1.0))
        h_loc = alto[np.clip(np.digitize(xp, bordes) - 1, 0, 39)]
        yp = A.Ly * (0.03 + h_loc * pseudoaleatorio(xp, d) ** 1.3)
        # las que están en x ≈ 0 (pared del fondo del modelo) se reparten por la cúpula del fundus
        cerca = xp < 0.3
        xd = np.where(cerca, xp - A.cap * 0.75 * pseudoaleatorio(d, xp + 1.0) * (1 - xp / 0.3), xp)
        P = A.punto(xd, yp, V, tau)
        o = np.argsort(-d)
        ax.scatter(P[o, 0], P[o, 1], s=1.5 + 6 * d[o], c=d[o], cmap=CMAP_D, norm=NORM_D, edgecolors="none", alpha=0.9, zorder=4)
    if rotulos:
        for x, y, txt in ((-1.0, 2.6, "fundus"), (2.5, 2.9, "cuerpo"), (7.0, 2.6, "antro")):
            p = A.punto(np.array(x), np.array(y), A.V_max, 0.0)
            ax.text(*p, txt, ha="center", va="center", fontsize=9, color=GRIS, style="italic", zorder=5,
                    path_effects=[patheffects.withStroke(linewidth=2.5, foreground="white", alpha=0.8)])
        ax.text(*(eje_e[-1] + [0.15, 0.25]), "esófago", fontsize=9, color=TINTA2, ha="center")
        ax.text(*(eje_d[-1] + [0.1, -0.45]), "duodeno", fontsize=9, color=TINTA2, ha="center")
        pp = A.punto(np.array(A.Lx), np.array(A.Ly * 1.1), V, tau)
        ax.text(pp[0] - 0.7, pp[1] + 0.55, "píloro", fontsize=9, color=TINTA2, ha="center")
    return ax


def texto_tiempo(t):
    m = int(t); s = int(round((t - m) * 60))
    if s == 60:
        m, s = m + 1, 0
    return f"{m // 60} h {m % 60:02d} min" if m >= 60 else f"{m} min {s:02d} s"


def etapa(t, ingesta):
    if t <= ingesta.duracion:
        return "INGESTA · la comida entra por el cardias en tragos"
    if t < 45:
        return "MEZCLA · el jugo diluye la comida desde las paredes"
    if t < 150:
        return "VACIADO · el antro tritura y el píloro deja pasar el quimo"
    return "FINAL · queda jugo ácido y restos que se vacían despacio"

## 4. Escala rápida: flujo LBM, mapas y calibración

Igual que en la fase 6: se carga el flujo de una onda (`campo_antro.npz`) o se regenera con LBM (≈ 75 s). Después se miden los mapas $D(x)$ y $\lambda_g(x)$ y el factor `escala_D`.

In [ ]:
CLAVES = ("t", "ux", "uy", "solido", "x", "y", "dx", "periodo", "nu")
datos = dict(np.load("campo_antro.npz")) if os.path.exists("campo_antro.npz") else {}
if any(c not in datos for c in CLAVES):
    print("Regenerando el flujo con LBM (≈ 75 s)...")
    exec(CODIGO_LBM, globals())
    pa = ParamAntro(); R = Red(np)
    S = simular_antro(pa, R, periodos=2, n_particulas=10, informar=False)
    t = S["t"]; k = int(round(pa.T / (t[1] - t[0]))); ult = slice(len(t) - k - 1, len(t)); dx = S["U"].dx
    datos = dict(t=t[ult] - t[ult][0], ux=np.nan_to_num(S["ux"][ult]), uy=np.nan_to_num(S["uy"][ult]), solido=S["solido"][ult],
                 x=(np.arange(S["nx"]) - 0.5) * dx, y=(np.arange(S["ny"]) + 0.5) * dx, dx=dx, periodo=pa.T, nu=pa.nu,
                 tau_max=np.nan_to_num(S["tau_max"][ult]))
    np.savez_compressed("campo_antro.npz", **datos)
CAMPO = CampoPeriodico(datos)
t0 = time.time()
MAPAS = mapas_escala_rapida(CAMPO, ondas=6, n=1500, ventana=2)
X = MAPAS["fluido"]["x_tray"]; idx = np.arange(0, X.shape[0], int(round(CAMPO.T / 0.5))); x0 = X[idx[0]]
rw = paseo_aleatorio(x0, MAPAS["fluido"]["Dx"], MAPAS["x"], CAMPO.T / 60, len(idx) - 1)
ESCALA_D = float(np.mean((X[idx[4]] - x0) ** 2) / np.mean((rw[4] - x0) ** 2))
print(f"Mapas y calibración: {time.time() - t0:.0f} s   ·   escala_D = {ESCALA_D:.2f}")

## 5. La simulación: de la primera cucharada a las 4 horas

Se guardan **196 instantáneas**:
- cada 5 s durante la ingesta (0–6 min);
- cada 30 s hasta los 10 min;
- cada 2 min hasta las 4 h.

In [ ]:
CONSENSO = Comida6("consenso", V_comida=315, prot_tampon=17, masa_solida=13, kcal=255)
PI = ParamInt(escala_D=ESCALA_D)
ING = Ingesta(duracion=6.0, tragos=18)
TIEMPOS = np.r_[np.arange(0, ING.duracion, 5 / 60), np.arange(ING.duracion, 10, 0.5), np.arange(10, 240.01, 2)]
t0 = time.time()
MOD = ModeloConIngesta(PI, CONSENSO, MAPAS, ING)
FOTOS = MOD.simular_con_instantaneas(TIEMPOS, dt=0.02)
H = MOD.hist; RET = MOD.retencion()
print(f"Simulación: {time.time() - t0:.0f} s, {len(FOTOS)} instantáneas")

# --- verificación ---
tot = MOD.totales()
err_M = abs(tot["M"] + MOD.sale["M"] - MOD.entra["M"]) / MOD.entra["M"]
err_s = abs(tot["solido"] + tot["cola"] + tot["P"] + MOD.sale["solido"] + MOD.sale["P"] - CONSENSO.masa_solida) / CONSENSO.masa_solida
M_max = max(float(f["M"].max()) for f in FOTOS)
VERIFICACION += [
    {"test": "V1 conservación del trazador de comida", "valor": err_M, "criterio": "< 1e-10", "pasa": err_M < 1e-10},
    {"test": "V2 conservación de la masa de sólido", "valor": err_s, "criterio": "< 1e-10", "pasa": err_s < 1e-10},
    {"test": "V3 la comida nunca está 'más concentrada' que pura", "valor": M_max, "criterio": "≤ 1.001", "pasa": M_max <= 1.001},
    {"test": "V4 el estómago se llena: V al acabar de comer [mL]", "valor": float(np.interp(ING.duracion, H["t"], H["V"])),
     "criterio": "250–340", "pasa": 250 <= np.interp(ING.duracion, H["t"], H["V"]) <= 340},
]
display(pd.DataFrame(VERIFICACION).style.format({"valor": "{:.3g}"}).hide(axis="index"))
ret_h = [float(np.interp(60 * h, H["t"], RET)) for h in (1, 2, 4)]
print("Retención de sólidos a 1/2/4 h: " + " / ".join(f"{r:.1f} %" for r in ret_h) + "   (rangos normales: 37–90 / 30–60 / 0–10 %)")

## 6. La forma: estómago en ayunas, lleno y con una onda en el antro

Una comprobación visual de la geometría antes de la película: el mismo dibujo con tres volúmenes y fases de onda distintos.

In [ ]:
ANAT = Anatomia(Lx=PI.Lx, Ly=PI.Ly, V_max=MOD.V_max)
fig, axs = plt.subplots(1, 3, figsize=(13, 5.2))
for a, (V, tau, tit) in zip(axs, [(25, 0.0, "En ayunas (25 mL)"), (MOD.V_max, 2.0, "Lleno, onda naciendo"),
                                  (MOD.V_max, 13.0, "Lleno, onda en el antro")]):
    a.set_aspect("equal"); a.axis("off"); a.set_xlim(-0.2, 11.6); a.set_ylim(0.6, 14.6)
    Xg, Yg, xs, ys = ANAT.malla(V, tau)
    a.pcolormesh(Xg, Yg, np.meshgrid(np.clip(xs, 0, None), ys)[0], shading="gouraud", cmap="Greys", vmin=-6, vmax=14)
    may, men = ANAT.contorno(V, tau)
    a.plot(*may.T, color=C2, lw=3, label="curvatura mayor (y = 0)"); a.plot(*men.T, color=C1, lw=3, label="curvatura menor (y = Ly)")
    for poli in (ANAT.esofago(V, tau)[0], ANAT.duodeno(V, tau)[0]):
        a.add_patch(Polygon(poli, closed=True, fc=TUBO, ec=PARED, lw=1.5))
    a.set_title(tit, color=GRIS)
axs[0].legend(loc="lower left", fontsize=8.5)
fig.suptitle("Geometría de la representación: la losa del modelo proyectada sobre una J (gris: coordenada x del modelo)", color=GRIS)
fig.savefig("figuras/01_geometria.png", dpi=200, bbox_inches="tight"); plt.show()

## 7. Ocho momentos clave

Lo mismo que la película, en una sola figura (el panel de dilución). Hay que fijarse en tres cosas:
- la comida **entra por arriba y se apila** en la curvatura mayor;
- el jugo forma un **anillo claro junto a las paredes** que se va comiendo la comida hacia el centro;
- el antro **tritura** las partículas (pasan de violeta a amarillo, más pequeñas) y las pequeñas **pasan al duodeno**.

In [ ]:
MOMENTOS = [0.5, 2.0, 6.0, 20.0, 45.0, 90.0, 150.0, 240.0]
TAU = reloj_ondas(FOTOS); T_FOTOS = np.array([f["t"] for f in FOTOS])
fig, axs = plt.subplots(2, 4, figsize=(15, 9.6))
for a, tm in zip(axs.ravel(), MOMENTOS):
    k = int(np.argmin(np.abs(T_FOTOS - tm)))
    dibujar_estomago(a, ANAT, FOTOS[k], TAU[k], "M", ING, rotulos=(tm == MOMENTOS[0]))
    a.set_title(f"{texto_tiempo(FOTOS[k]['t'])}   ·   {FOTOS[k]['V']:.0f} mL", fontsize=11, color=GRIS)
    a.text(0.5, 0.02, etapa(FOTOS[k]["t"], ING).split(" · ")[0], transform=a.transAxes, ha="center", fontsize=9, color=TINTA2)
fig.subplots_adjust(right=0.9, wspace=0.02, hspace=0.12)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=NORM_M, cmap=CMAP_M), cax=fig.add_axes([0.915, 0.55, 0.012, 0.33]))
cb.set_label("fracción de comida en el líquido")
cb = fig.colorbar(plt.cm.ScalarMappable(norm=NORM_D, cmap=CMAP_D), cax=fig.add_axes([0.915, 0.12, 0.012, 0.33]))
cb.set_label("diámetro de partícula [mm]")
fig.suptitle("Ingesta, mezcla, trituración y vaciado de la comida de consenso", fontsize=14, color=GRIS, y=0.97)
fig.savefig("figuras/02_momentos.png", dpi=170, bbox_inches="tight"); plt.show()

## 8. La película

Hay dos paneles con la misma geometría deformada:
- **dilución**: fracción de comida en el líquido;
- **acidez**: pH local.

Debajo, la evolución del sólido retenido, de la comida en el líquido y del volumen, con un cursor. Se guarda en `figuras/03_digestion_2D.mp4` (y `.gif` si `GUARDAR_GIF = True`) y se muestra aquí.

In [ ]:
def montar_figura(fotos, hist, ret, A, ingesta):
    fig = plt.figure(figsize=(12.5, 8.4))
    gs = fig.add_gridspec(2, 4, height_ratios=[4.2, 1.0], width_ratios=[1, 0.035, 1, 0.035], hspace=0.08, wspace=0.05,
                          left=0.03, right=0.95, top=0.89, bottom=0.07)
    axM, axP = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 2])
    cbM = fig.colorbar(plt.cm.ScalarMappable(norm=NORM_M, cmap=CMAP_M), cax=fig.add_subplot(gs[0, 1]))
    cbM.set_label("fracción de comida en el líquido  (1 = comida, 0 = jugo)")
    cbP = fig.colorbar(plt.cm.ScalarMappable(norm=NORM_PH, cmap=CMAP_PH), cax=fig.add_subplot(gs[0, 3]))
    cbP.set_label("pH local")
    axT = fig.add_subplot(gs[1, :])
    t = hist["t"]
    axT.plot(t, ret, color=C2, lw=1.8, label="sólido en el estómago [% de la comida]")
    axT.plot(t, 100 * hist["frac_comida_media"], color=C4, lw=1.8, label="comida en el líquido [%]")
    axT.plot(t, 100 * hist["V"] / A.V_max, color=C1, lw=1.5, ls="--", label="volumen [% del máximo]")
    axT.set_xlim(0, t[-1]); axT.set_ylim(0, 105); axT.set_xlabel("tiempo [min]")
    axT.axvspan(0, ingesta.duracion, color=C4, alpha=0.15, lw=0)
    axT.legend(loc="upper right", fontsize=8.5, ncol=3)
    cursor = axT.axvline(0, color=GRIS, lw=1.2)
    titulo = fig.text(0.03, 0.955, "", fontsize=14, weight="bold", color=GRIS)
    sub = fig.text(0.03, 0.918, "", fontsize=10.5, color=TINTA2)
    vel = fig.text(0.95, 0.955, "", fontsize=9.5, color=TINTA2, ha="right")
    fig.text(0.95, 0.925, "partículas: color y tamaño = diámetro (0–4 mm)", fontsize=9, color=TINTA2, ha="right")
    axM.set_title("Dilución de la comida", fontsize=11.5, color=GRIS)
    axP.set_title("Acidez (pH)", fontsize=11.5, color=GRIS)
    tau = reloj_ondas(fotos)
    tt = np.array([f["t"] for f in fotos])

    def dibujar(k):
        f = fotos[k]
        dibujar_estomago(axM, A, f, tau[k], "M", ingesta, rotulos=True)
        dibujar_estomago(axP, A, f, tau[k], "pH", ingesta)
        axM.set_title("Dilución de la comida", fontsize=11.5, color=GRIS); axP.set_title("Acidez (pH)", fontsize=11.5, color=GRIS)
        cursor.set_xdata([f["t"], f["t"]])
        titulo.set_text(f"t = {texto_tiempo(f['t'])}   ·   V = {f['V']:.0f} mL")
        sub.set_text(etapa(f["t"], ingesta))
        dt_real = (tt[min(k + 1, len(tt) - 1)] - tt[max(k - 1, 0)]) / 2 * 60 if len(tt) > 1 else 0
        vel.set_text(f"cámara rápida ×{dt_real / 0.1:,.0f}" + ("   ·   ondas en tiempo real" if f["t"] <= ingesta.duracion
                     else "   ·   ondas: fase ilustrativa"))
        return []
    return fig, dibujar

In [ ]:
GUARDAR_GIF = True          # el GIF tarda ~1 min más y pesa más; el MP4 es suficiente para ver la película
FPS = 10
fig, dibujar = montar_figura(FOTOS, H, RET, ANAT, ING)
ANIM = animation.FuncAnimation(fig, dibujar, frames=len(FOTOS), interval=1000 / FPS, blit=False)
t0 = time.time()
RUTA_MP4 = "figuras/03_digestion_2D.mp4"
try:
    ANIM.save(RUTA_MP4, writer=animation.FFMpegWriter(fps=FPS, bitrate=2400), dpi=100)
    print(f"{RUTA_MP4}  ({os.path.getsize(RUTA_MP4) / 1e6:.1f} MB, {time.time() - t0:.0f} s)")
except Exception as e:
    RUTA_MP4 = None; print(f"mp4: no disponible ({type(e).__name__}); se mostrará con JavaScript")
if GUARDAR_GIF:
    try:
        t0 = time.time(); ANIM.save("figuras/03_digestion_2D.gif", writer=animation.PillowWriter(fps=FPS), dpi=60)
        print(f"figuras/03_digestion_2D.gif  ({os.path.getsize('figuras/03_digestion_2D.gif') / 1e6:.1f} MB, {time.time() - t0:.0f} s)")
    except Exception as e:
        print(f"gif: no disponible ({type(e).__name__})")
plt.close(fig)
if RUTA_MP4:
    with open(RUTA_MP4, "rb") as fh:
        b64 = base64.b64encode(fh.read()).decode()
    display(HTML(f'<video width="900" controls loop autoplay muted><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    display(HTML(ANIM.to_jshtml()))

## 9. Lo que muestra la película, en números

La película se puede comprobar con datos. Se miden tres tiempos característicos:
- cuándo la comida del líquido baja a la mitad (**dilución**);
- cuándo el pH de la concentración media de H⁺ cruza 3 (**acidificación**). Es una media de $[\mathrm H^+]$, así que pesa mucho el jugo ácido de las paredes; el núcleo de comida sigue a pH ≈ 5 bastante más tiempo, como se ve en la película;
- cuándo se vacía la mitad del sólido ($t_{1/2}$).

In [ ]:
t = H["t"]; fc = H["frac_comida_media"]; tras = t > ING.duracion
def cruce(y, nivel):
    i = np.where(tras & (y <= nivel))[0]
    return float(t[i[0]]) if i.size else np.nan
pico = float(fc[tras].max())
RESUMEN = pd.DataFrame([
    {"magnitud": "volumen máximo [mL]", "valor": float(H["V"].max()), "referencia": "≈ 315 + 25 mL ingeridos menos lo ya vaciado"},
    {"magnitud": "comida en el líquido al acabar de comer [%]", "valor": 100 * pico, "referencia": "el resto es jugo en ayunas y secretado"},
    {"magnitud": "dilución a la mitad (comida en el líquido) [min]", "valor": cruce(fc, pico / 2), "referencia": "—"},
    {"magnitud": "pH de la [H⁺] media ≤ 3 [min]", "valor": cruce(H["pH_medio"], 3.0), "referencia": "—"},
    {"magnitud": "t½ vaciado de sólidos [min]", "valor": cruce(RET, 50.0), "referencia": "≈ 60–130 min (gammagrafía, comida estándar)"},
    {"magnitud": "retención 4 h [%]", "valor": float(RET[-1]), "referencia": "0–10 % (normal)"},
])
display(RESUMEN.style.format({"valor": "{:.1f}"}).hide(axis="index"))

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
axs[0].plot(t, H["V"], color=C1); axs[0].set(title="Volumen", xlabel="t [min]", ylabel="mL")
axs[1].plot(t, 100 * fc, color=C4, label="comida en el líquido"); axs[1].plot(t, RET, color=C2, label="sólido retenido")
axs[1].set(title="Dilución y vaciado", xlabel="t [min]", ylabel="%"); axs[1].legend()
axs[2].fill_between(t, H["pH_min"], H["pH_max"], color=C5, alpha=0.2, lw=0, label="rango en el estómago")
axs[2].plot(t, H["pH_medio"], color=C5, label="pH de la [H⁺] media"); axs[2].set(title="Acidez", xlabel="t [min]", ylabel="pH"); axs[2].legend()
for a in axs:
    a.axvspan(0, ING.duracion, color=C4, alpha=0.15, lw=0)
fig.tight_layout(); fig.savefig("figuras/04_series.png", dpi=170, bbox_inches="tight"); plt.show()

## 10. Pruebas

In [ ]:
for v in VERIFICACION:
    assert v["pasa"], v["test"]
assert len(FOTOS) == len(TIEMPOS)
assert FOTOS[0]["d"].size == 0 and FOTOS[0]["V"] == CONSENSO.V_basal          # empieza en ayunas
assert FOTOS[-1]["salidas"] or MOD.sale["solido"] > 0                          # algo ha pasado el píloro
assert all(np.isfinite(f["pH"]).all() and np.isfinite(f["M"]).all() for f in FOTOS)
# la deformación peristáltica: la onda estrecha el antro pero nunca cierra la luz del todo
w_min = min(ANAT.anchura(np.linspace(0, 10, 400), MOD.V_max, tau).min() for tau in np.arange(0, 20, 0.5))
assert 0 < w_min
print(f"Todas las pruebas pasan  ·  anchura mínima dibujada {w_min:.2f}")

## 11. Conclusiones y límites

**Qué se ve**
- **Ingesta (0–6 min)**: el estómago pasa de 25 a unos 315 mL. El fundus y el cuerpo se ensanchan y el antro casi no cambia. Las partículas caen y se apilan en la curvatura mayor del fundus y el cuerpo. El líquido de la comida tampona el contenido hasta pH ≈ 5,5.
- **Mezcla (≈ 6–45 min)**: el jugo entra por las paredes del cuerpo y forma un **anillo ácido** alrededor de un núcleo de comida que sigue tamponado. Es la "estratificación" que describen los estudios de resonancia: la comida se diluye desde fuera hacia dentro y el pH ácido avanza con ella.
- **Vaciado (≈ 45–150 min)**: las partículas avanzan hacia el antro, las ondas las trituran y las pequeñas pasan al duodeno con el líquido. El color del duodeno muestra lo diluido y lo ácido que sale el quimo.
- **Final**: queda jugo ácido y un resto de partículas que se vacían despacio.

**Lo que hay que tener en cuenta**
- **La forma es un dibujo.** La física es la de una losa 2D de 10 × 4 cm con espesor uniforme. La "J", la cúpula del fundus, el esófago y el duodeno sirven para leer el resultado en anatomía, no son un cálculo en esa geometría.
- **Las ondas del dibujo son cinemáticas.** Usan los mismos parámetros que el LBM y su efecto (mezcla y molienda) está en el modelo a través de los mapas promediados en fase. Pero el modelo no ve cada onda individual, y tras la ingesta su fase en la película es ilustrativa.
- **La ingesta añade una suposición**: la comida entra a caudal constante por el cardias y sus sólidos en 18 tragos. Por eso el vaciado de sólidos empieza más tarde que en la fase 6, donde la comida estaba toda dentro en $t = 0$. Con ingesta, la retención queda en ≈ 99 / 59 / 13 % a 1 / 2 / 4 h. La de 2 h está dentro del rango normal (en la fase 6 no lo estaba). Las de 1 h y 4 h quedan algo por encima, es decir, un vaciado un poco lento. Es una diferencia del orden de la incertidumbre paramétrica de la fase 6, sobre todo en $\gamma$ (cuánto arrastra la pared a las partículas). **No se ha recalibrado nada para esta fase**: la película usa el modelo tal como quedó validado.
- **Siguiente paso natural**: un acoplamiento directo LBM–transporte en GPU durante los primeros minutos (la ingesta). Ahí la geometría cambia deprisa y el promediado en fase es menos fiable.